In [ ]:
import sys
sys.path.append("../../src/")

from logger import CrawlLogger
logger = CrawlLogger("inspect_scraper").logger

from bs4 import BeautifulSoup
import requests
import re
import json
import time
import os
from Scraper import YT_Scraper
from Exceptions import *
from VideoData import VideoData
import ast

In [ ]:
scraper = YT_Scraper()

In [ ]:
id = "vZQRTKf8nrg"

page = scraper.get_yt_page(id)
with open("test_page.html", "w", encoding="utf-8") as f:
    f.write(page.text)

In [ ]:
soup = BeautifulSoup(page.content, "html.parser")

data_block = None
for script in soup.find_all("script"):
    if script.string and "ytInitialData" in script.string:
        match = re.search(r"ytInitialData\s*=\s*({.*?});", script.string, re.DOTALL)
        if match:
            data_str = match.group(1)
            try:
                data_block = json.loads(data_str)
            except json.JSONDecodeError as e:
                raise ParsingError(data_str, details={
                    "error": str(e),
                    "page": page.text
                })
            break

with open("data_block.json", "w", encoding="utf-8") as f:
    json.dump(data_block, f, ensure_ascii=False, indent=2)


In [ ]:
# convert dirty datablock

if os.path.exists("data_block_dirty.json"):
    with open("data_block_dirty.json", "r", encoding="utf-8") as f:
        block = ast.literal_eval(f.read())

    with open("data_block_dirty_parsed.json", "w", encoding="utf-8") as f:
        json.dump(block, f, ensure_ascii=False, indent=2)

In [ ]:
# results_block = data_block["contents"]["twoColumnWatchNextResults"]["secondaryResults"]["secondaryResults"]["results"]
results_block = data_block["contents"]["twoColumnWatchNextResults"]["secondaryResults"]["secondaryResults"]["results"]

with open("video_block.json", "w", encoding="utf-8") as f:
    json.dump(results_block, f, ensure_ascii=False, indent=2)

In [ ]:
def extract_videodata_from_block(block) -> VideoData:
    if "endScreenVideoRenderer" not in block:
            return None
    
    video_data = block["endScreenVideoRenderer"]

    video_id = video_data.get("videoId")
    if video_id is None:
        return None

    title = video_data.get("title", {}).get("simpleText")
    
    views = video_data.get("shortViewCountText", {}).get("simpleText")
    if views is None:
        views = video_data.get("shortViewCountText", {}).get("runs", [{}])[0].get("text")
        
    length = video_data.get("lengthInSeconds")
    
    channel = video_data.get("shortBylineText", {}).get("runs", [{}])[0].get("text")
    
    uploaded_str = video_data.get("publishedTimeText", {}).get("simpleText")
    
    

    if views is not None:
        try:
            views_str = views
            if "No views" in views_str:
                views_str = "0"

            views_str = views_str.split(" ")[0].strip()
            views_str = views_str.replace(",", ".")
            
            if "K" in views_str:
                views = int(float(views_str.replace("K", "")) * 1e3)
            elif "M" in views_str:
                views = int(float(views_str.replace("M", "")) * 1e6)
            elif "B" in views_str:
                views = int(float(views_str.replace("B", "")) * 1e9)
            else:
                views = int(views_str.replace(".", "").replace(",", ""))
            
        except Exception as e:
            logger.error(f"Error while parsing views. Original views string: {views}. Error: {e}")
            views = None
            

    # NOTE: if length is None, video might be a live stream
    
    # if length is not None:
    #     try:
    #         total_seconds = 0
    #         parts = length.strip().split(":")[::-1]
    #         for i, part in enumerate(parts):
    #             total_seconds += int(part) * (60 ** i)
    #         length = total_seconds
    #     except Exception as e:
    #         logger.error(f"Error while parsing length. Original length string: {length}. Error: {e}")
    #         length = None
    

    if uploaded_str is not None:
        try:
            uploaded_digits = re.sub(r"[^\d]", "", uploaded_str)
            if uploaded_digits.isdigit():
                uploaded_digits = int(uploaded_digits)

                if "second" in uploaded_str:
                    uploaded = uploaded_digits
                elif "minute" in uploaded_str:
                    uploaded = uploaded_digits * 60
                elif "hour" in uploaded_str:
                    uploaded = uploaded_digits * 3600
                elif "day" in uploaded_str:
                    uploaded = uploaded_digits * 86400
                elif "week" in uploaded_str:
                    uploaded = uploaded_digits * 604800
                elif "month" in uploaded_str:
                    uploaded = uploaded_digits * 2592000
                elif "year" in uploaded_str:
                    uploaded = uploaded_digits * 31536000
                else:
                    logger.error(f"Unknown time unit in uploaded string: {uploaded_str}")
                    uploaded = None

                if uploaded is not None:
                    uploaded = int(time.time()) - uploaded

            else:
                logger.error(f"Error while parsing uploaded time. Expected an integer, got {uploaded_digits}. Original uploaded string: {uploaded_str}")
                uploaded = None
        except Exception as e:
            logger.error(f"Error while parsing uploaded time. Original uploaded string: {uploaded_str}. Error: {e}")
            uploaded = None


    # print(f"Video ID: {video_id}")
    # print(f"Title: {title}")
    # print(f"Channel: {channel}")
    # print(f"Length (seconds): {length}")
    # print(f"Views: {views}")
    # print(f"Uploaded: {uploaded} (original string: {uploaded_str})")

    return VideoData(
        video_id=video_id,
        title=title,
        channel=channel,
        views=views,
        length=length,
        uploaded=uploaded
    )
    
    

def filter_recommendation_data(page, id) -> list[VideoData]:
    soup = BeautifulSoup(page.content, "html.parser")
    
    data_block = None
    for script in soup.find_all("script"):
        if script.string and "ytInitialData" in script.string:
            match = re.search(r"ytInitialData\s*=\s*({.*?});", script.string, re.DOTALL)
            if match:
                data_str = match.group(1)
                try:
                    data_block = json.loads(data_str)
                except json.JSONDecodeError as e:
                    raise ParsingError(data_str, details={
                        "error": str(e),
                        "video_id": id,
                        "page": page.text
                    })
                break

    if data_block is None:
        raise NoDataFound("Couldn't find ytInitialData in the page.", details={
            "video_id": id,
            "page": page.text
        })
    
    try:
        # if video is unavailable, return empty recommendations
        current_video = data_block["contents"]["twoColumnWatchNextResults"]["results"]
        if "video unavailable" in json.dumps(current_video, ensure_ascii=False).lower():
            logger.warning(f"Video {id} is unavailable. Returning empty recommendations.")
            return []
    except Exception as e:
        pass

    try:
        # results_block = data_block["contents"]["twoColumnWatchNextResults"]["secondaryResults"]["secondaryResults"]["results"]
        results_block = data_block["playerOverlays"]["playerOverlayRenderer"]["endScreen"]["watchNextEndScreenRenderer"]
        if "results" not in results_block:
            logger.warning(f"No recommendations found for {id} in the end screen.")
            return []
        results_block = results_block["results"]
    except Exception as e:
        raise UnexpectedFormat("Couldn't find the recommended videos in the page.", details={
            "error": str(e),
            "video_id": id,
            "data_block": data_block
        })

    next_videos = []
    try:
        for entry in results_block:
            video_data = extract_videodata_from_block(entry)
            if video_data is not None:
                next_videos.append(video_data)
            
    except Exception as e:
        raise UnexpectedFormat("Couldn't find the video data in the page.", details={
            "error": str(e),
            "video_id": id,
            "page": page.text,
            "data_block": data_block
        })

    return next_videos

In [ ]:
results = filter_recommendation_data(page, id)
for vid in results:
    print(vid)